# Scenarios / Quality Measurement
A published **example measure** run by name over FHIR resources: the per-patient results, the rate as one SQL
cell, a follow-up question asked of the same helper library the measure uses, and the same measure over another
period.

**Nothing is authored in this notebook.** The measure is not a string in a cell: the deploy published it, with
the libraries it includes and its value sets, into the default environment's knowledge, and it runs **by
name**. That is the point: a measure is content you publish and version, not code you paste. No CQL on
Databricks App, no registry, no UI — the compiler, the environment, and Spark.

The example measure is a hypertension control measure with an initial population, exclusions and a numerator,
over 4,000 synthetic patients, about 1,000 synthesized for each example measure. In a real measure everything is
larger but the shape is the same.

**Runs on serverless — no cluster to attach.** Just **Run All**.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

## 1. The data: the default environment
Every run is in one environment: a Unity Catalog catalog whose `data` schema holds one table per FHIR type,
named for the type (`condition`), each returning the resource as a `resource` struct and `patient_id`, the
key of the patient the row belongs to. The deploy builds the default environment with the example patients, so there is
nothing to load here. The engine lowers every `[Resource]` retrieve to a read of its table, and the plan
filters above it.

The read schema is **the compiler's, not inferred**: each resource has the struct the compiler's FHIR model
defines (`client.resource_schema("Condition")` shows it). In your own environment the tables, or views over your
governed FHIR tables, hold the same shape; the CQL does not change.

**Expected output:** the environment's name and catalog, then one line per type: its row count and the table
it is read from.

In [0]:
from cql_databricks.resolvers import EnvironmentResolver
from cql_databricks.data_bindings import spark_runner

environment = EnvironmentResolver(spark_runner(spark)).default()
print("environment:", environment.name, "in catalog", environment.catalog)
for t in ("Condition", "Encounter", "Patient"):
    view = f"{environment.schema('data')}.{t.lower()}"
    print(f"{t:12} {spark.table(view).count()} rows -> {view}")

### Is the data readable?
Before running a measure over it, one `%%cql` cell asks the data a direct question: self-contained
CQL, no dependencies. The cell declares the question, and the Python cell after it runs it.

**Expected output:** 4,000 rows, one per patient, each with true or false for whether they have any condition
on record — a direct check to verify the data is readable.

In [0]:
%%cql
library MountCheck version '1.0.0'
using FHIR version '4.0.1'

context Patient

define "Has A Condition": exists ([Condition])

In [0]:
display(cql(library="MountCheck").exec("Has A Condition"))

## 2. The measure: published in the environment's knowledge
This is the piece that makes a *published* measure runnable rather than a pasted one.

The example measure `include`s a helper library, `MeasureCommon`, and `FHIRHelpers`, and the compile service
resolves **none** of them — by design. It is a dumb endpoint: the caller resolves the `include` closure and
ships every dependency; the service only merges what it is handed. That purity boundary is why the service
reads no filesystem, no registry and no network — and why the resolver lives in the client.

A client from `connect()` with nothing mounted reads libraries by name from the environment's knowledge, where
the deploy published the example measures. **Getting Started / Overview** shows the other sources: libraries
in cells and in a folder.

**Expected output:** the schema the measure's libraries are read from.

In [0]:
from cql_databricks.compiler import connect

client = connect(spark)
print("libraries are read by name from", environment.schema("knowledge"))

## 3. Bind the measure by name, and read it before running it
`client.cql(library="HypertensionBPControl")` addresses the measure by **name**. Resolving it walks
the `include` closure, so the common libraries and helper functions are found and shipped with the
compile without being mentioned here.

An `include` that states a version resolves only to that version; a version nothing declares fails loud,
naming the versions it did find.

The handle is lazy and the compile is memoized, so inspecting and then running costs one round trip.
No data has been touched at this point.

**Expected output:**
```
populations : ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
defines     : [... 14 defines including population definitions, hypertension logic and blood pressure
           criteria ...]
```
`populations` finds the eCQM populations by their well-known names; it is what `exec()` runs when you name
no targets. The measure has four populations: initial population, denominator, exclusions and numerator.
Over the 2026-01-01 to 2026-12-31 period, 590 of 4,000 patients are in the initial population.

In [0]:
measure = client.cql(library="HypertensionBPControl") \
                .with_measurement_period("2026-01-01", "2026-12-31")

print("populations :", measure.populations())
print("defines     :", measure.defines())

## 4. Run it
`exec()` with the population defines evaluates them together, compiling the shared graph
**once** — the measures' logic flows through each other, and that shared work is computed once.

Every define's result arrives with **identical schemas** (`__subject_id`, `__val`), because a define names the
object and never a column. The frames are presented side by side for each patient so you can
see the pattern: which patients have hypertension, which are in scope, which are excluded, and which meet control.

**Expected output:** 4,000 rows, one per patient. 590 are in the initial population (adults 18–85 with
hypertension and a qualifying visit). Of those, 543 are not excluded (not in hospice or palliative care, not
pregnant, no kidney failure). Of those 543, 337 have controlled blood pressure — a rate of about 62%.

In [0]:
from pyspark.sql import functions as F

from cql_databricks.compiler import VALUE_COLUMN

defines = measure.exec("Initial Population", "Denominator", "Denominator Exclusions", "Numerator")
# Name each result for the SQL cell in section 5. A named library's run reads its views under a prefix of its own.
for name, df in defines.items():
    df.createOrReplaceTempView(name.replace(" ", "_"))
report = (defines["Initial Population"].withColumnRenamed(VALUE_COLUMN, "initial_population")
          .join(defines["Denominator"].withColumnRenamed(VALUE_COLUMN, "denominator"), "__subject_id")
          .join(defines["Denominator Exclusions"].withColumnRenamed(VALUE_COLUMN, "excluded"), "__subject_id")
          .join(defines["Numerator"].withColumnRenamed(VALUE_COLUMN, "numerator"), "__subject_id")
          .orderBy("__subject_id"))
report.display()

## 5. Query the result as SQL
The cell above named each define's frame as a session temp view, with spaces as underscores
(`Initial_Population`, `Denominator`, etc.), so the headline rate is one SQL cell away — no CQL, no Python, nothing to export.

**Expected output:** one row — the rate of controlled blood pressure among the denominator (after exclusions).

Each define is true or false on its own, so the query nests them the way CQF Measures scores a proportion: the
numerator counts only patients in the denominator and not excluded.

In [0]:
%sql
SELECT count_if(d.__val) AS denominator,
       count_if(d.__val AND NOT coalesce(ex.__val, false)) AS denominator_after_exclusions,
       count_if(d.__val AND NOT coalesce(ex.__val, false) AND n.__val) AS numerator
FROM Denominator d
LEFT JOIN Denominator_Exclusions ex ON d.__subject_id = ex.__subject_id
LEFT JOIN Numerator n ON d.__subject_id = n.__subject_id

## 6. Ask a question the measure did not answer
A care-gap report always provokes a follow-up: *which* patients, and why are they excluded? The published
definitions in the helper libraries are addressable, so the follow-up can be asked of **the same libraries the measure uses**
rather than re-implemented in ad-hoc SQL — which is where consistency usually breaks, someone
rebuilding "age at end of period" slightly differently and getting a number nobody can reconcile.

The follow-up here asks the helper library the measure includes, `MeasureCommon`, one of the
questions behind its exclusions: who was in hospice care during the period?

**Where an `include` is found.** A `%%cql` cell's `include` resolves against earlier cells, then a mounted
folder, then the environment's knowledge, where `MeasureCommon` is, so a cell can `include` it too. Here the
follow-up runs the helper library by name with `library=`.

**Expected output:** 4,000 rows, one per patient. 42 are true: they had hospice care during 2026, or were
discharged to hospice in it.

In [0]:
followup = client.cql(library="MeasureCommon").with_measurement_period("2026-01-01", "2026-12-31")
followup.exec("In Hospice Care").orderBy("__subject_id").display()

## 7. The same measure, a different period
The measurement period is a `parameter`, and the handle is immutable-fluent — `with_measurement_period`
returns a NEW handle rather than rebinding this one. Nothing about the content changes; the question
is asked of a different year.

Watch the population turn over. For 2026 the measure reported 590 patients in the initial population; for 2025
there are 0 — every hypertension patient's qualifying visit in this dataset is in 2026.

That is worth sitting with: **a rate is meaningless without its period**, and the same content
honestly produces both numbers.

Each run of a named library reads its views under a prefix of its own, so this run leaves the
frames from section 4, and the views section 5 read, on 2026. One session, one library, two
parameterizations, and neither run disturbs the other.

**Expected output:**
```
2025: initial population = 0
```

In [0]:
prior = measure.with_measurement_period("2025-01-01", "2025-12-31").exec("Initial Population")
initial_pop_2025 = prior.collect()
count_2025 = sum(1 for r in initial_pop_2025 if r[VALUE_COLUMN])
print(f"2025: initial population = {count_2025}")

---
### What just happened
A measure published in the environment's knowledge, its dependencies resolved by name, its data read through
one contract — and a per-patient result, a SQL count, an ad-hoc follow-up asking the helper library the same
question, and the same measure re-parameterized to a different year, all producing honest numbers that cannot
disagree with each other. No application was involved.

**Getting Started / Overview** ends by reading a measure without running it: the define graph, the ELM and the
Spark plan the compiler produces before any data is touched. Back to **Start Here** for the index.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "scenarios/quality_measurement"}))